In [ ]:
# v1
# v2 : added "if it likes it then has a higher chnace of liking it" (i assume randomness too much)
# v3 : changed it to +-0.1 pet like dislike (slow but nice randomness was too much i think?)
# v4 : just a combined plot +-1 yes
# v5 : sigmoid for liking chance normalization for friendliness, +0.3 -0.5 , and decay of 0.5% per tick 

In [2]:
import csv
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go


##
df = pd.read_csv('activity_log.csv', header=None, 
                 names=['timestamp' , 'activity_name', 'pets_involved'])
df["pet_name"] = df['pets_involved'].str.split(";", n=1).str[0]
df["partner_name"] = df['pets_involved'].str.split(";", n=1).str[-1] # i'll process this later
df['partner_name'] = df['partner_name'].where(df['partner_name'] != df['pet_name'], None)  # later
df_activity_finished = df
# df_activity_finished['timestamp'] = pd.to_datetime(df_activity_finished['timestamp'], unit="ms")
df_activity_finished["partner_name"] = (
    df_activity_finished["partner_name"]
    .str.split("@", n=1)
    .str[-1] # I love you python
)

##
df = pd.read_csv('relationships.csv', header=None,
                 names=['timestamp' , 'pet_name', 'other_entity_id', 'friendliness'])
df_relationships = df
df_relationships['timestamp'] = pd.to_datetime(df_relationships['timestamp'], unit="ms")
df_relationships["other_entity_id"] = (
    df_relationships["other_entity_id"]
    .str.split("@", n=1)
    .str[-1]
)

In [3]:
sum(s in "alice -> mom" for s in ["alice", "brice", "ant"]) # i love you python pt 2

1

In [4]:
df = df_activity_finished.copy()
df = df.dropna(subset=["pet_name", "partner_name"])
df["pair_key"] = df.apply(
    lambda r: tuple(sorted((r.pet_name, r.partner_name))),
    axis=1
)

df = df.sort_values(["pair_key", "activity_name", "timestamp"])
pd.DataFrame(df.groupby(["pair_key", "activity_name"])["timestamp"].diff())#.lt(100))
df_activity_finished = df

In [5]:
pet_names = df_relationships["pet_name"].unique()
df_relationships['pair'] = df_relationships.apply(lambda row: f"{row['pet_name']} → {row['other_entity_id']}", axis=1)
df_relationships_pivot = df_relationships.pivot(
    index='pair',
    columns='timestamp',
    values='friendliness'
)

df_relationships_pivot = df_relationships_pivot.fillna(0)

In [7]:
mask = df_relationships_pivot.index.map(
    lambda a: (sum(s in a for s in pet_names) >= 2)
)

heat = df_relationships_pivot[mask]
heat = heat.sort_index(axis=1)

import plotly.express as px

fig = px.imshow(
    heat,
    aspect="auto",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    labels={
        "x": "Time",
        "y": "Relationship",
        "color": "Value"
    }
)

group_size = 5 

for i in range(group_size, len(heat.index), group_size):
    fig.add_hline(
        y=i - 0.5,
        line_width=5,
        line_color="black"
    )

row_to_y = {row: i for i, row in enumerate(heat.index)}
print(row_to_y)
line_x = []
line_y = []

marker_x = []
marker_y = []
marker_hover = []

df_activity_finished['timestamp'] = pd.to_datetime(df_activity_finished['timestamp'], unit="ms")

for _, row in df_activity_finished.iterrows():
    pet1, pet2 = sorted(row["pets_involved"].split(";"))

    r1 = f"{pet1} → {pet2}"
    r2 = f"{pet2} → {pet1}"

    if r1 not in row_to_y or r2 not in row_to_y:
        continue

    y1 = row_to_y[r1]
    y2 = row_to_y[r2]
    y1 = r1
    y2 = r2
    x = row["timestamp"]


    line_x += [x, x, None]
    line_y += [y1, y2, None]

    marker_x += [x, x]
    marker_y += [y1, y2]

    marker_hover += [
        f"{pet1} → {pet2}<br>{row['activity_name']}",
        f"{pet2} → {pet1}<br>{row['activity_name']}",
    ]

# fig.add_trace(go.Scatter(
#     x=line_x,
#     y=line_y,
#     mode="lines",
#     line=dict(color="black", width=1),
#     hoverinfo="skip",
#     showlegend=False,
# ))

fig.add_trace(go.Scatter(
    x=marker_x,
    y=marker_y,
    mode="markers",
    text=marker_hover,
    hovertemplate="%{text}<extra></extra>",
    marker=dict(size=3, color='black', symbol='diamond'),
    showlegend=False,
))
fig.update_layout(height=900)

fig.show()

{'Ant → Bat': 0, 'Ant → Bee': 1, 'Ant → alice': 2, 'Ant → bear': 3, 'Ant → brice': 4, 'Bat → Ant': 5, 'Bat → Bee': 6, 'Bat → alice': 7, 'Bat → bear': 8, 'Bat → brice': 9, 'Bee → Ant': 10, 'Bee → Bat': 11, 'Bee → alice': 12, 'Bee → bear': 13, 'Bee → brice': 14, 'alice → Ant': 15, 'alice → Bat': 16, 'alice → Bee': 17, 'alice → bear': 18, 'alice → brice': 19, 'bear → Ant': 20, 'bear → Bat': 21, 'bear → Bee': 22, 'bear → alice': 23, 'bear → brice': 24, 'brice → Ant': 25, 'brice → Bat': 26, 'brice → Bee': 27, 'brice → alice': 28, 'brice → bear': 29}


In [ ]:
df["activity_pair"] = df["pet_name"] + " : " + df["activity_name"]

heat = df.pivot_table(
    index="activity_pair",
    columns="timestamp",
    values="activity_relationship",
    aggfunc="last"
)

heat = heat.sort_index(axis=1).ffill(axis=1)
heat.fillna(0, inplace=True)

fig = px.imshow(
    heat,
    aspect="auto",
    color_continuous_scale="RdYlGn",
)

fig.update_layout(height=900)
fig.show()